<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/RAG_Conversacional_Memoria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Conversacional + Memoria

Notebook para Google Colab basado en el RAG del notebook original.

Flujo:

PDF → chunks → embeddings → FAISS

Pregunta → memoria → reformulación → FAISS → contexto → GPT → respuesta → memoria

La memoria permite preguntas como:
- ¿Qué es Machine Learning?
- ¿Y cuáles son sus tipos?
- ¿Cuál de ellos utiliza datos etiquetados?


## Celda 1 - INSTALAR LIBRERÍAS


In [1]:
# ============================================================

!pip install -q pypdf openai sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 62.3 MB/s eta 0:00:00


## Celda 2 - CARGAR API KEY


In [2]:
# ============================================================

from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "No se encontró OPENAI_API_KEY en Google Colab Secrets."
    )

print("API Key cargada correctamente.")




API Key cargada correctamente.


## Celda 3 - CLIENTE OPENAI


In [3]:
# ============================================================

from openai import OpenAI

client = OpenAI(
    api_key=api_key
)

## Celda 4 - SUBIR PDF


In [4]:
# ============================================================

from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise ValueError("No se seleccionó ningún archivo PDF.")

pdf_path = list(uploaded.keys())[0]

print("PDF cargado:", pdf_path)

Saving documento_prueba_ia_pdf_machine_learning.pdf to documento_prueba_ia_pdf_machine_learning.pdf
PDF cargado: documento_prueba_ia_pdf_machine_learning.pdf


## Celda 5 - EXTRAER TEXTO DEL PDF


In [5]:
# ============================================================

from pypdf import PdfReader

reader = PdfReader(pdf_path)

paginas = []

for numero, pagina in enumerate(reader.pages, start=1):

    texto = pagina.extract_text()

    if texto and texto.strip():

        paginas.append({
            "pagina": numero,
            "texto": texto
        })

print("Páginas procesadas:", len(paginas))




Páginas procesadas: 3


## Celda 6 - CREAR CHUNKS


In [6]:
# ============================================================

chunks = []

tamano_chunk = 800

for pagina in paginas:

    texto = pagina["texto"]

    for i in range(
        0,
        len(texto),
        tamano_chunk
    ):

        fragmento = texto[i:i + tamano_chunk]

        if fragmento.strip():

            chunks.append({
                "pagina": pagina["pagina"],
                "texto": fragmento
            })

print("Fragmentos creados:", len(chunks))




Fragmentos creados: 6


## Celda 7 - GENERAR EMBEDDINGS


In [7]:
# ============================================================

from sentence_transformers import SentenceTransformer

model_embedding = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

textos = [
    chunk["texto"]
    for chunk in chunks
]

embeddings = model_embedding.encode(
    textos,
    show_progress_bar=True
)

print(
    "Embeddings generados:",
    len(embeddings)
)




modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generados: 6


## Celda 8 - CREAR ÍNDICE FAISS


In [8]:
# ============================================================

import faiss
import numpy as np

embeddings = np.array(
    embeddings,
    dtype="float32"
)

index = faiss.IndexFlatL2(
    embeddings.shape[1]
)

index.add(embeddings)

print(
    "Vectores almacenados en FAISS:",
    index.ntotal
)




Vectores almacenados en FAISS: 6


## Celda 9 - MEMORIA DE CONVERSACIÓN


In [9]:
# ============================================================

memoria_conversacion = []

print("Memoria de conversación creada.")




Memoria de conversación creada.


## Celda 10 - GUARDAR MENSAJE


In [10]:
# ============================================================

def guardar_mensaje(rol, contenido):

    memoria_conversacion.append({
        "rol": rol,
        "contenido": contenido
    })




## Celda 11 - OBTENER HISTORIAL


In [11]:
# ============================================================

def obtener_historial(max_mensajes=6):

    mensajes = memoria_conversacion[-max_mensajes:]

    historial = ""

    for mensaje in mensajes:

        historial += (
            f"{mensaje['rol'].upper()}: "
            f"{mensaje['contenido']}\n\n"
        )

    return historial




## Celda 12 - MOSTRAR MEMORIA


In [12]:
# ============================================================

def mostrar_memoria():

    print("\n")
    print("=" * 60)
    print("MEMORIA DE CONVERSACIÓN")
    print("=" * 60)

    if not memoria_conversacion:

        print("La memoria está vacía.")
        return

    for mensaje in memoria_conversacion:

        print(
            f"{mensaje['rol'].upper()}:"
        )

        print(
            mensaje["contenido"]
        )

        print("-" * 60)




## Celda 13 - REFORMULAR PREGUNTA


In [13]:
# ============================================================

def reformular_pregunta(pregunta):

    historial = obtener_historial()

    prompt = f"""
Tenemos una conversación sobre un documento PDF.

HISTORIAL DE CONVERSACIÓN:

{historial}

NUEVA PREGUNTA DEL USUARIO:

{pregunta}

Determina si la nueva pregunta depende
del contexto anterior.

Si depende del contexto anterior,
reformula la pregunta para que sea
completamente independiente y pueda
utilizarse para buscar información
en el PDF.

Si NO depende del contexto anterior,
conserva la pregunta prácticamente igual.

REGLAS:

- No respondas la pregunta.
- No agregues información que no aparezca
  en la conversación.
- Solamente devuelve la pregunta reformulada.
- Responde en español.
"""

    respuesta = client.responses.create(

        model="gpt-5.4-mini",

        instructions="""
Eres un reformulador de preguntas para
un sistema RAG conversacional.

Tu única tarea es convertir preguntas
dependientes del contexto en preguntas
independientes.

NO respondas la pregunta.

Devuelve solamente la pregunta reformulada.
""",

        input=prompt
    )

    return respuesta.output_text.strip()




## Celda 14 - BUSCAR EN FAISS


In [14]:
# ============================================================

def buscar_pdf(pregunta, k=5):

    vector = model_embedding.encode(
        [pregunta]
    )

    vector = np.array(
        vector,
        dtype="float32"
    )

    distancias, indices = index.search(
        vector,
        k
    )

    resultados = []

    for distancia, indice in zip(
        distancias[0],
        indices[0]
    ):

        if indice < 0:
            continue

        resultados.append({
            "pagina": chunks[indice]["pagina"],
            "texto": chunks[indice]["texto"],
            "distancia": float(distancia)
        })

    return resultados




## Celda 15 - OBTENER CONTEXTO DEL PDF


In [15]:
# ============================================================

def obtener_contexto(pregunta, k=5):

    resultados = buscar_pdf(
        pregunta,
        k=k
    )

    contexto = ""

    for i, resultado in enumerate(
        resultados
    ):

        contexto += f"""
--- FRAGMENTO {i + 1} ---

Página:
{resultado['pagina']}

Texto:
{resultado['texto']}

"""

    return contexto, resultados




## Celda 16 - RAG CONVERSACIONAL


In [16]:
# ============================================================

def preguntar_rag(pregunta, k=5):

    # --------------------------------------------------------
    # 1. Guardar pregunta original
    # --------------------------------------------------------

    guardar_mensaje(
        "usuario",
        pregunta
    )


    # --------------------------------------------------------
    # 2. Reformular considerando la conversación
    # --------------------------------------------------------

    pregunta_reformulada = (
        reformular_pregunta(
            pregunta
        )
    )


    # --------------------------------------------------------
    # 3. Buscar en el PDF
    # --------------------------------------------------------

    contexto, resultados = (
        obtener_contexto(
            pregunta_reformulada,
            k=k
        )
    )


    # --------------------------------------------------------
    # 4. Obtener historial
    # --------------------------------------------------------

    historial = obtener_historial()


    # --------------------------------------------------------
    # 5. Construir prompt
    # --------------------------------------------------------

    prompt = f"""

HISTORIAL DE CONVERSACIÓN:

{historial}

PREGUNTA ORIGINAL DEL USUARIO:

{pregunta}

PREGUNTA REFORMULADA PARA BUSCAR EN EL PDF:

{pregunta_reformulada}

CONTEXTO RECUPERADO DEL PDF:

{contexto}

INSTRUCCIONES:

1. Responde utilizando únicamente
   la información del contexto recuperado.

2. Utiliza el historial solamente para
   comprender el contexto de la conversación.

3. No inventes información.

4. Si el PDF no contiene información
   suficiente para responder, dilo claramente.

5. No utilices conocimiento externo.

6. Responde en español.

7. Explica la respuesta de forma clara.

8. Si es necesario, utiliza listas.
"""


    # --------------------------------------------------------
    # 6. Generar respuesta
    # --------------------------------------------------------

    respuesta = client.responses.create(

        model="gpt-5.4-mini",

        instructions="""
Eres un asistente especializado
en análisis documental.

Responde preguntas sobre un documento PDF
utilizando únicamente la información
recuperada del documento.

Puedes utilizar el historial para comprender
referencias como:

- eso
- ellos
- sus tipos
- el segundo
- lo anterior
- ese concepto
- dicha técnica

Nunca debes utilizar conocimiento externo
al documento recuperado.
""",

        input=prompt
    )

    texto_respuesta = (
        respuesta.output_text.strip()
    )


    # --------------------------------------------------------
    # 7. Guardar respuesta
    # --------------------------------------------------------

    guardar_mensaje(
        "ia",
        texto_respuesta
    )


    # --------------------------------------------------------
    # 8. Obtener páginas
    # --------------------------------------------------------

    paginas_utilizadas = sorted(
        set(
            resultado["pagina"]
            for resultado in resultados
        )
    )


    # --------------------------------------------------------
    # 9. Devolver resultados
    # --------------------------------------------------------

    return {

        "respuesta":
            texto_respuesta,

        "pregunta_original":
            pregunta,

        "pregunta_reformulada":
            pregunta_reformulada,

        "paginas":
            paginas_utilizadas,

        "resultados":
            resultados
    }




## Celda 17 - CHAT INTERACTIVO


In [ ]:
# ============================================================

print()
print("=" * 60)
print("       RAG CONVERSACIONAL + MEMORIA")
print("=" * 60)
print()
print("Escribe 'salir' para terminar.")
print("Escribe 'memoria' para ver el historial.")
print("Escribe 'limpiar' para reiniciar la conversación.")
print()


while True:

    pregunta = input("\nUsuario: ").strip()


    # --------------------------------------------------------
    # SALIR
    # --------------------------------------------------------

    if pregunta.lower() == "salir":

        print(
            "\nConversación terminada."
        )

        break


    # --------------------------------------------------------
    # MOSTRAR MEMORIA
    # --------------------------------------------------------

    if pregunta.lower() == "memoria":

        mostrar_memoria()

        continue


    # --------------------------------------------------------
    # LIMPIAR MEMORIA
    # --------------------------------------------------------

    if pregunta.lower() == "limpiar":

        memoria_conversacion.clear()

        print(
            "\nMemoria reiniciada."
        )

        continue


    # --------------------------------------------------------
    # PREGUNTA VACÍA
    # --------------------------------------------------------

    if not pregunta:

        print(
            "Escribe una pregunta."
        )

        continue


    # --------------------------------------------------------
    # EJECUTAR RAG
    # --------------------------------------------------------

    try:

        resultado = preguntar_rag(
            pregunta,
            k=5
        )


        # ----------------------------------------------------
        # RESPUESTA
        # ----------------------------------------------------

        print()
        print("-" * 60)

        print("IA:")

        print(
            resultado["respuesta"]
        )


        # ----------------------------------------------------
        # PÁGINAS
        # ----------------------------------------------------

        print()

        print(
            "Páginas consultadas:"
        )

        print(
            ", ".join(
                str(p)
                for p in resultado["paginas"]
            )
        )


    except Exception as e:

        print()
        print(
            "Ocurrió un error:"
        )

        print(e)





       RAG CONVERSACIONAL + MEMORIA

Escribe 'salir' para terminar.
Escribe 'memoria' para ver el historial.
Escribe 'limpiar' para reiniciar la conversación.


Usuario: ¿Qué es Machine Learning?

------------------------------------------------------------
IA:
Según el documento, **Machine Learning** o **aprendizaje automático** es **una rama de la inteligencia artificial que permite que un sistema aprenda patrones a partir de datos y utilice esos patrones para realizar predicciones o tomar decisiones**.

Además, el documento menciona que un flujo básico de Machine Learning suele incluir:

- recopilación de datos
- limpieza
- exploración
- selección de variables
- entrenamiento
- evaluación
- despliegue del modelo

Si quieres, también puedo decirte **cuáles son los tres tipos principales de aprendizaje** que menciona el PDF.

Páginas consultadas:
1, 2, 3

Usuario: ¿Cuáles son sus aplicaciones?

------------------------------------------------------------
IA:
Según el PDF, las aplicac

## Celda 18 - FUNCIÓN PARA REINICIAR MEMORIA


In [18]:
# ============================================================

def limpiar_memoria():

    memoria_conversacion.clear()

    print(
        "La memoria de conversación fue limpiada."
    )




## Celda 19 - INFORMACIÓN DEL SISTEMA


In [19]:
# ============================================================

print()
print("=" * 60)
print("INFORMACIÓN DEL RAG")
print("=" * 60)

print(
    "PDF:",
    pdf_path
)

print(
    "Páginas procesadas:",
    len(paginas)
)

print(
    "Fragmentos:",
    len(chunks)
)

print(
    "Vectores FAISS:",
    index.ntotal
)

print(
    "Modelo embeddings:",
    "all-MiniLM-L6-v2"
)

print(
    "Modelo LLM:",
    "gpt-5.4-mini"
)

print(
    "Mensajes en memoria:",
    len(memoria_conversacion)
)



INFORMACIÓN DEL RAG
PDF: documento_prueba_ia_pdf_machine_learning.pdf
Páginas procesadas: 3
Fragmentos: 6
Vectores FAISS: 6
Modelo embeddings: all-MiniLM-L6-v2
Modelo LLM: gpt-5.4-mini
Mensajes en memoria: 4
